## Características da geração

Os dados foram construídos para apresentar comportamentos analíticos realistas, incluindo:

- crescimento das vendas;
- sazonalidade;
- concentração de faturamento;
- diferenças de desempenho entre vendedores;
- clientes recorrentes;
- clientes novos;
- clientes em risco de abandono;
- produtos de alto e baixo giro;
- ruptura e excesso de estoque;
- inadimplência.

Também são introduzidos propositalmente pequenos problemas de qualidade nos arquivos brutos.

Esses problemas serão identificados e tratados posteriormente na camada Silver.

## Reprodutibilidade

A geração utiliza uma semente aleatória fixa (`SEMENTE = 42`), permitindo reproduzir o mesmo conjunto de dados em novas execuções.

> Todos os nomes, empresas, clientes, produtos e registros presentes neste projeto são fictícios.

In [0]:
# Bibliotecas utilizadas na geração dos dados

import random
from datetime import date, timedelta
from collections import defaultdict

from pyspark.sql import functions as F


# ---------------------------------------------------------
# Parâmetros gerais
# ---------------------------------------------------------

SEMENTE = 42

random.seed(SEMENTE)

DATA_INICIO = date(2024, 1, 1)
DATA_FIM = date(2026, 7, 31)


# ---------------------------------------------------------
# Identifica o catálogo do workspace
# ---------------------------------------------------------

catalogo_atual = spark.sql(
    "SELECT current_catalog()"
).first()[0]


# Caminho criado no notebook anterior

caminho_volume = (
    f"/Volumes/{catalogo_atual}/varejo_landing/dados_brutos"
)


print(f"Catálogo: {catalogo_atual}")
print(f"Período: {DATA_INICIO} até {DATA_FIM}")
print(f"Volume: {caminho_volume}")
print(f"Semente: {SEMENTE}")

In [0]:
# ---------------------------------------------------------
# FORNECEDORES
# ---------------------------------------------------------

estados_fornecedores = [
    "GO", "SP", "MG", "PR",
    "SC", "DF", "MT", "MS"
]

pesos_estados = [
    28, 22, 12, 10,
    7, 8, 7, 6
]

prefixos_fornecedor = [
    "Alvorada",
    "Aurora",
    "Central",
    "Horizonte",
    "Integra",
    "Nova Era",
    "Planalto",
    "Pioneira",
    "União",
    "Vale"
]

tipos_fornecedor = [
    "Comercial",
    "Distribuição",
    "Indústria",
    "Suprimentos",
    "Atacado"
]


fornecedores = []


for i in range(1, 81):

    fornecedores.append(
        {
            "id_fornecedor": f"FOR{i:04d}",

            "nome_fornecedor": (
                f"{random.choice(prefixos_fornecedor)} "
                f"{random.choice(tipos_fornecedor)} "
                f"{i:03d}"
            ),

            "estado": random.choices(
                estados_fornecedores,
                weights=pesos_estados,
                k=1
            )[0],

            "prazo_medio_entrega": random.randint(2, 15),

            "condicao_pagamento": random.choice(
                [
                    "15 dias",
                    "30 dias",
                    "45 dias",
                    "60 dias"
                ]
            ),

            "avaliacao_fornecedor": round(
                random.uniform(3.0, 5.0),
                2
            ),

            "situacao_fornecedor": (
                "Ativo"
                if random.random() < 0.96
                else "Inativo"
            )
        }
    )


print(f"Fornecedores gerados: {len(fornecedores):,}")

In [0]:
# ---------------------------------------------------------
# PRODUTOS
# ---------------------------------------------------------

categorias = {

    "Alimentos": [
        "Mercearia",
        "Biscoitos",
        "Massas",
        "Conservas",
        "Temperos"
    ],

    "Bebidas": [
        "Refrigerantes",
        "Sucos",
        "Águas",
        "Energéticos",
        "Cafés"
    ],

    "Higiene": [
        "Higiene pessoal",
        "Papel higiênico",
        "Cuidados pessoais"
    ],

    "Limpeza": [
        "Limpeza doméstica",
        "Lavanderia",
        "Descartáveis"
    ],

    "Papelaria": [
        "Escritório",
        "Escolar",
        "Papel"
    ],

    "Utilidades": [
        "Cozinha",
        "Organização",
        "Uso geral"
    ]
}


peso_categoria = {
    "Alimentos": 28,
    "Bebidas": 20,
    "Higiene": 15,
    "Limpeza": 15,
    "Papelaria": 10,
    "Utilidades": 12
}


faixa_custo = {

    "Alimentos": (3, 45),
    "Bebidas": (2, 30),
    "Higiene": (3, 50),
    "Limpeza": (3, 65),
    "Papelaria": (1, 35),
    "Utilidades": (4, 90)
}


faixa_margem = {

    "Alimentos": (0.12, 0.28),
    "Bebidas": (0.10, 0.24),
    "Higiene": (0.18, 0.35),
    "Limpeza": (0.18, 0.38),
    "Papelaria": (0.22, 0.45),
    "Utilidades": (0.20, 0.42)
}


marcas = [
    "Aquarela",
    "BemMais",
    "Cerrado",
    "Essencial",
    "Ideal",
    "Nobre",
    "Primor",
    "Viva",
    "Sol",
    "Verde"
]


fornecedores_ativos = [
    f["id_fornecedor"]
    for f in fornecedores
    if f["situacao_fornecedor"] == "Ativo"
]


lista_categorias = list(categorias.keys())

pesos_categorias = [
    peso_categoria[categoria]
    for categoria in lista_categorias
]


produtos = []


for i in range(1, 501):

    categoria = random.choices(
        lista_categorias,
        weights=pesos_categorias,
        k=1
    )[0]

    subcategoria = random.choice(
        categorias[categoria]
    )

    custo = round(
        random.uniform(
            *faixa_custo[categoria]
        ),
        2
    )

    margem = random.uniform(
        *faixa_margem[categoria]
    )

    preco = round(
        custo / (1 - margem),
        2
    )

    marca = random.choice(marcas)

    produtos.append(
        {
            "id_produto": f"PRO{i:04d}",

            "nome_produto": (
                f"{subcategoria} "
                f"{marca} "
                f"{i:03d}"
            ),

            "categoria": categoria,

            "subcategoria": subcategoria,

            "marca": marca,

            "unidade_medida": random.choice(
                ["UN", "CX", "PCT", "FD"]
            ),

            "custo_unitario": custo,

            "preco_venda": preco,

            "margem_padrao": round(
                (preco - custo) / preco,
                4
            ),

            "id_fornecedor_principal": random.choice(
                fornecedores_ativos
            ),

            "situacao_produto": (
                "Ativo"
                if random.random() < 0.97
                else "Inativo"
            )
        }
    )


print(f"Produtos gerados: {len(produtos):,}")

In [0]:
# ---------------------------------------------------------
# POTENCIAL DE DEMANDA DOS PRODUTOS
# ---------------------------------------------------------

produtos_ativos = [
    p["id_produto"]
    for p in produtos
    if p["situacao_produto"] == "Ativo"
]


# Embaralha os produtos antes de atribuir o potencial
# de demanda para evitar relação com o ID.

ordem_demanda = produtos_ativos.copy()

random.shuffle(ordem_demanda)


demanda_base = {}

quantidade_produtos = len(ordem_demanda)


for posicao, id_produto in enumerate(
    ordem_demanda,
    start=1
):

    # Cria uma distribuição desigual de demanda.
    #
    # Poucos produtos terão demanda muito alta,
    # enquanto muitos produtos apresentarão
    # demanda intermediária ou baixa.

    proporcao = (
        quantidade_produtos - posicao + 1
    ) / quantidade_produtos

    demanda = (
        2
        + 58 * (proporcao ** 2.6)
    )

    demanda_base[id_produto] = demanda


print(
    "Potencial médio de demanda:",
    round(
        sum(demanda_base.values())
        / len(demanda_base),
        2
    )
)

In [0]:
# ---------------------------------------------------------
# VENDEDORES
# ---------------------------------------------------------

primeiros_nomes = [
    "Ana", "Bruno", "Camila", "Daniel",
    "Eduardo", "Fernanda", "Gabriel",
    "Helena", "Igor", "Juliana",
    "Lucas", "Mariana", "Nicolas",
    "Patricia", "Rafael", "Renata",
    "Thiago", "Vanessa", "William",
    "Yasmin"
]


sobrenomes = [
    "Almeida",
    "Barbosa",
    "Carvalho",
    "Dias",
    "Ferreira",
    "Gomes",
    "Lima",
    "Martins",
    "Oliveira",
    "Pereira",
    "Ribeiro",
    "Santos",
    "Silva",
    "Souza",
    "Teixeira"
]


regioes = [
    "Goiânia e Região Metropolitana",
    "Entorno do DF",
    "Sul de Goiás",
    "Sudoeste de Goiás",
    "Norte de Goiás"
]


vendedores = []

# Variável interna que será utilizada
# para criar diferenças de desempenho.

fator_desempenho_vendedor = {}


for i in range(1, 21):

    nivel = random.choices(
        ["Júnior", "Pleno", "Sênior"],
        weights=[5, 9, 6],
        k=1
    )[0]

    fator_nivel = {
        "Júnior": 0.82,
        "Pleno": 1.00,
        "Sênior": 1.18
    }[nivel]

    id_vendedor = f"VEN{i:03d}"

    vendedores.append(
        {
            "id_vendedor": id_vendedor,

            "nome_vendedor": (
                f"{primeiros_nomes[i - 1]} "
                f"{random.choice(sobrenomes)}"
            ),

            "data_admissao": (
                date(2021, 1, 1)
                + timedelta(
                    days=random.randint(0, 1000)
                )
            ),

            "regiao": regioes[
                (i - 1) % len(regioes)
            ],

            "nivel": nivel,

            "situacao_vendedor": "Ativo"
        }
    )

    fator_desempenho_vendedor[id_vendedor] = (
        fator_nivel
        * random.uniform(0.88, 1.15)
    )


print(f"Vendedores gerados: {len(vendedores):,}")

In [0]:
# ---------------------------------------------------------
# CLIENTES
# ---------------------------------------------------------

cidades = [

    (
        "Goiânia",
        "GO",
        "Goiânia e Região Metropolitana",
        20
    ),

    (
        "Aparecida de Goiânia",
        "GO",
        "Goiânia e Região Metropolitana",
        10
    ),

    (
        "Anápolis",
        "GO",
        "Goiânia e Região Metropolitana",
        8
    ),

    (
        "Senador Canedo",
        "GO",
        "Goiânia e Região Metropolitana",
        5
    ),

    (
        "Trindade",
        "GO",
        "Goiânia e Região Metropolitana",
        5
    ),

    (
        "Rio Verde",
        "GO",
        "Sudoeste de Goiás",
        8
    ),

    (
        "Jataí",
        "GO",
        "Sudoeste de Goiás",
        5
    ),

    (
        "Catalão",
        "GO",
        "Sul de Goiás",
        5
    ),

    (
        "Itumbiara",
        "GO",
        "Sul de Goiás",
        4
    ),

    (
        "Luziânia",
        "GO",
        "Entorno do DF",
        7
    ),

    (
        "Águas Lindas de Goiás",
        "GO",
        "Entorno do DF",
        6
    ),

    (
        "Formosa",
        "GO",
        "Entorno do DF",
        4
    ),

    (
        "Porangatu",
        "GO",
        "Norte de Goiás",
        2
    ),

    (
        "Goianésia",
        "GO",
        "Norte de Goiás",
        3
    ),

    (
        "Brasília",
        "DF",
        "Entorno do DF",
        8
    )
]


segmentos = [
    "Supermercado",
    "Mercearia",
    "Padaria",
    "Restaurante",
    "Farmácia",
    "Loja de conveniência",
    "Escritório"
]


pesos_segmentos = [
    14, 26, 14, 16,
    8, 10, 12
]


prefixo_estabelecimento = {

    "Supermercado": "Supermercado",
    "Mercearia": "Mercearia",
    "Padaria": "Panificadora",
    "Restaurante": "Restaurante",
    "Farmácia": "Drogaria",
    "Loja de conveniência": "Conveniência",
    "Escritório": "Empresa"
}


nomes_fantasia = [
    "Cerrado",
    "Ipê",
    "Primavera",
    "Central",
    "Boa Compra",
    "Ponto Certo",
    "Sol Nascente",
    "Vila Nova",
    "Imperial",
    "Brasil",
    "Aliança",
    "Estação"
]


# Perfis internos utilizados para criar
# comportamentos diferentes no RFM.

perfis = [
    "Campeão",
    "Leal",
    "Regular",
    "Em risco",
    "Perdido",
    "Novo"
]

pesos_perfis = [
    5,
    15,
    55,
    10,
    5,
    10
]


# Organiza os vendedores por região.

vendedores_por_regiao = defaultdict(list)

for vendedor in vendedores:

    vendedores_por_regiao[
        vendedor["regiao"]
    ].append(
        vendedor["id_vendedor"]
    )


clientes = []

# Metadados utilizados somente pelo gerador.
# Eles NÃO serão gravados na fonte de clientes.

perfil_interno_cliente = {}


for i in range(1, 2001):

    segmento = random.choices(
        segmentos,
        weights=pesos_segmentos,
        k=1
    )[0]

    cidade = random.choices(
        cidades,
        weights=[
            item[3]
            for item in cidades
        ],
        k=1
    )[0]

    porte = random.choices(
        ["Pequeno", "Médio", "Grande"],
        weights=[66, 28, 6],
        k=1
    )[0]

    perfil = random.choices(
        perfis,
        weights=pesos_perfis,
        k=1
    )[0]


    # Clientes novos entram mais recentemente.

    if perfil == "Novo":

        data_cadastro = (
            date(2025, 10, 1)
            + timedelta(
                days=random.randint(
                    0,
                    (
                        date(2026, 6, 30)
                        - date(2025, 10, 1)
                    ).days
                )
            )
        )

    else:

        data_cadastro = (
            date(2021, 1, 1)
            + timedelta(
                days=random.randint(
                    0,
                    (
                        date(2025, 9, 30)
                        - date(2021, 1, 1)
                    ).days
                )
            )
        )


    faixa_limite = {

        "Pequeno": (5000, 30000),

        "Médio": (
            30000,
            100000
        ),

        "Grande": (
            100000,
            300000
        )

    }[porte]


    limite_credito = round(
        random.uniform(*faixa_limite),
        -2
    )


    # Define quando clientes perdidos ou em risco
    # deixam de comprar.

    if perfil == "Perdido":

        data_limite_compra = (
            date(2025, 1, 1)
            + timedelta(
                days=random.randint(0, 364)
            )
        )

    elif perfil == "Em risco":

        data_limite_compra = (
            date(2026, 2, 1)
            + timedelta(
                days=random.randint(0, 100)
            )
        )

    else:

        data_limite_compra = None


    fator_perfil = {

        "Campeão": 3.2,
        "Leal": 1.9,
        "Regular": 1.0,
        "Em risco": 1.6,
        "Perdido": 1.1,
        "Novo": 0.9

    }[perfil]


    fator_porte = {

        "Pequeno": 0.7,
        "Médio": 1.5,
        "Grande": 3.2

    }[porte]


    fator_segmento = {

        "Supermercado": 1.35,
        "Mercearia": 0.90,
        "Padaria": 1.00,
        "Restaurante": 1.05,
        "Farmácia": 0.75,
        "Loja de conveniência": 0.85,
        "Escritório": 0.65

    }[segmento]


    peso_compra = max(
        0.1,

        fator_perfil
        * fator_porte
        * fator_segmento
        * random.uniform(0.65, 1.35)
    )


    risco_credito = random.choices(
        ["Baixo", "Médio", "Alto"],
        weights=[65, 27, 8],
        k=1
    )[0]


    id_vendedor_responsavel = random.choice(
        vendedores_por_regiao[
            cidade[2]
        ]
    )


    id_cliente = f"CLI{i:05d}"


    clientes.append(
        {
            "id_cliente": id_cliente,

            "nome_cliente": (
                f"{prefixo_estabelecimento[segmento]} "
                f"{random.choice(nomes_fantasia)} "
                f"{i:04d}"
            ),

            "segmento_cliente": segmento,

            "cidade": cidade[0],

            "estado": cidade[1],

            "data_cadastro": data_cadastro,

            "porte_cliente": porte,

            "limite_credito": float(
                limite_credito
            ),

            "situacao_cliente": (
                "Inativo"
                if (
                    perfil == "Perdido"
                    and random.random() < 0.45
                )
                else "Ativo"
            )
        }
    )


    perfil_interno_cliente[id_cliente] = {

        "perfil": perfil,

        "peso": peso_compra,

        "data_limite_compra":
            data_limite_compra,

        "risco_credito":
            risco_credito,

        "vendedor":
            id_vendedor_responsavel,

        "porte": porte,

        "segmento": segmento,

        "data_cadastro":
            data_cadastro
    }


print(f"Clientes gerados: {len(clientes):,}")

In [0]:
# ---------------------------------------------------------
# CALENDÁRIO E SAZONALIDADE
# ---------------------------------------------------------

datas = [
    DATA_INICIO + timedelta(days=i)
    for i in range(
        (DATA_FIM - DATA_INICIO).days + 1
    )
]


fator_mes = {

    1: 0.95,
    2: 0.90,
    3: 1.00,
    4: 1.02,
    5: 1.04,
    6: 1.00,
    7: 1.03,
    8: 1.05,
    9: 1.08,
    10: 1.12,
    11: 1.20,
    12: 1.35
}


pesos_datas = []


for data in datas:

    # Crescimento gradual ao longo do tempo.

    crescimento = (
        1
        + 0.00045
        * (data - DATA_INICIO).days
    )


    # Menor movimento aos finais de semana.

    if data.weekday() < 5:

        fator_dia_semana = 1.0

    elif data.weekday() == 5:

        fator_dia_semana = 0.55

    else:

        fator_dia_semana = 0.15


    peso = (
        crescimento
        * fator_mes[data.month]
        * fator_dia_semana
    )


    pesos_datas.append(peso)


print(
    f"Dias disponíveis para geração: "
    f"{len(datas):,}"
)

In [0]:
# ---------------------------------------------------------
# DISTRIBUIÇÃO DE PRODUTOS
# ---------------------------------------------------------

produto_por_categoria = defaultdict(list)

peso_produto_por_categoria = defaultdict(list)


mapa_produtos = {
    produto["id_produto"]: produto
    for produto in produtos
}


for produto in produtos:

    if produto["situacao_produto"] != "Ativo":
        continue

    id_produto = produto["id_produto"]

    categoria = produto["categoria"]

    produto_por_categoria[
        categoria
    ].append(
        id_produto
    )

    peso_produto_por_categoria[
        categoria
    ].append(
        demanda_base[id_produto]
    )

In [0]:
preferencias_segmento = {

    "Supermercado": {
        "Alimentos": 32,
        "Bebidas": 25,
        "Higiene": 13,
        "Limpeza": 13,
        "Papelaria": 5,
        "Utilidades": 12
    },

    "Mercearia": {
        "Alimentos": 38,
        "Bebidas": 28,
        "Higiene": 10,
        "Limpeza": 10,
        "Papelaria": 4,
        "Utilidades": 10
    },

    "Padaria": {
        "Alimentos": 42,
        "Bebidas": 28,
        "Higiene": 6,
        "Limpeza": 12,
        "Papelaria": 3,
        "Utilidades": 9
    },

    "Restaurante": {
        "Alimentos": 42,
        "Bebidas": 22,
        "Higiene": 5,
        "Limpeza": 18,
        "Papelaria": 3,
        "Utilidades": 10
    },

    "Farmácia": {
        "Alimentos": 10,
        "Bebidas": 10,
        "Higiene": 42,
        "Limpeza": 15,
        "Papelaria": 8,
        "Utilidades": 15
    },

    "Loja de conveniência": {
        "Alimentos": 30,
        "Bebidas": 40,
        "Higiene": 8,
        "Limpeza": 6,
        "Papelaria": 5,
        "Utilidades": 11
    },

    "Escritório": {
        "Alimentos": 10,
        "Bebidas": 12,
        "Higiene": 8,
        "Limpeza": 20,
        "Papelaria": 35,
        "Utilidades": 15
    }
}

In [0]:
# ---------------------------------------------------------
# FUNÇÕES AUXILIARES
# ---------------------------------------------------------

ids_clientes = [
    cliente["id_cliente"]
    for cliente in clientes
]


pesos_clientes = [
    perfil_interno_cliente[id_cliente]["peso"]
    for id_cliente in ids_clientes
]


def escolher_cliente(data_venda):

    """
    Seleciona um cliente elegível para realizar
    uma compra na data informada.
    """

    for _ in range(25):

        id_cliente = random.choices(
            ids_clientes,
            weights=pesos_clientes,
            k=1
        )[0]

        perfil = perfil_interno_cliente[
            id_cliente
        ]


        cadastrado = (
            perfil["data_cadastro"]
            <= data_venda
        )


        ainda_compra = (
            perfil["data_limite_compra"]
            is None

            or data_venda
            <= perfil["data_limite_compra"]
        )


        if cadastrado and ainda_compra:

            return id_cliente


    # Fallback de segurança.

    elegiveis = [

        id_cliente

        for id_cliente in ids_clientes

        if (
            perfil_interno_cliente[
                id_cliente
            ]["data_cadastro"]
            <= data_venda

            and (
                perfil_interno_cliente[
                    id_cliente
                ]["data_limite_compra"]
                is None

                or data_venda
                <= perfil_interno_cliente[
                    id_cliente
                ]["data_limite_compra"]
            )
        )
    ]


    return random.choice(elegiveis)

In [0]:
def escolher_categoria(
    segmento,
    mes
):

    """
    Seleciona uma categoria considerando
    o perfil do cliente e a sazonalidade.
    """

    preferencias = (
        preferencias_segmento[
            segmento
        ].copy()
    )


    # Final do ano:
    # aumento de bebidas e alimentos.

    if mes in [11, 12]:

        preferencias["Bebidas"] *= 1.25

        preferencias["Alimentos"] *= 1.15


    # Janeiro e fevereiro:
    # aumento de papelaria.

    if mes in [1, 2]:

        preferencias["Papelaria"] *= 1.35


    categorias_disponiveis = list(
        preferencias.keys()
    )


    return random.choices(

        categorias_disponiveis,

        weights=[
            preferencias[categoria]
            for categoria
            in categorias_disponiveis
        ],

        k=1

    )[0]

In [0]:
def gerar_quantidade(
    porte,
    categoria
):

    """
    Define a quantidade comprada conforme
    o porte do cliente.
    """

    faixas = {

        "Pequeno": (1, 8),

        "Médio": (4, 24),

        "Grande": (10, 60)
    }


    minimo, maximo = faixas[porte]


    quantidade = random.randint(
        minimo,
        maximo
    )


    if categoria in [
        "Alimentos",
        "Bebidas"
    ]:

        quantidade = int(
            quantidade
            * random.uniform(
                1.1,
                1.6
            )
        )


    return max(
        quantidade,
        1
    )

In [0]:
# ---------------------------------------------------------
# VENDAS
# ---------------------------------------------------------

vendas_validas = []

pedidos_venda = []


ids_vendedores = [
    vendedor["id_vendedor"]
    for vendedor in vendedores
]


QUANTIDADE_PEDIDOS = 40_000


for numero_pedido in range(
    1,
    QUANTIDADE_PEDIDOS + 1
):

    data_venda = random.choices(
        datas,
        weights=pesos_datas,
        k=1
    )[0]


    id_cliente = escolher_cliente(
        data_venda
    )


    cliente = perfil_interno_cliente[
        id_cliente
    ]


    # Na maior parte dos casos,
    # o cliente compra com seu vendedor
    # responsável.

    if random.random() < 0.90:

        id_vendedor = cliente[
            "vendedor"
        ]

    else:

        id_vendedor = random.choice(
            ids_vendedores
        )


    quantidade_itens = random.choices(

        [1, 2, 3, 4, 5, 6],

        weights=[
            10,
            18,
            25,
            20,
            16,
            11
        ],

        k=1

    )[0]


    produtos_utilizados = set()

    total_pedido = 0


    for _ in range(
        quantidade_itens
    ):

        # Evita repetir o mesmo produto
        # dentro de um pedido.

        for tentativa in range(10):

            categoria = escolher_categoria(
                cliente["segmento"],
                data_venda.month
            )


            id_produto = random.choices(

                produto_por_categoria[
                    categoria
                ],

                weights=
                    peso_produto_por_categoria[
                        categoria
                    ],

                k=1

            )[0]


            if (
                id_produto
                not in produtos_utilizados
            ):

                break


        produtos_utilizados.add(
            id_produto
        )


        produto = mapa_produtos[
            id_produto
        ]


        quantidade = gerar_quantidade(
            cliente["porte"],
            categoria
        )


        # Pequena variação do preço
        # ao longo das vendas.

        preco_unitario = round(

            produto["preco_venda"]

            * random.uniform(
                0.97,
                1.05
            ),

            2
        )


        # Clientes maiores recebem,
        # em média, descontos maiores.

        faixa_desconto = {

            "Pequeno": (
                0.00,
                0.06
            ),

            "Médio": (
                0.02,
                0.10
            ),

            "Grande": (
                0.04,
                0.15
            )

        }[cliente["porte"]]


        percentual_desconto = (
            random.uniform(
                *faixa_desconto
            )
        )


        valor_bruto = round(
            quantidade
            * preco_unitario,
            2
        )


        valor_desconto = round(
            valor_bruto
            * percentual_desconto,
            2
        )


        valor_liquido = round(
            valor_bruto
            - valor_desconto,
            2
        )


        custo_total = round(

            quantidade

            * produto[
                "custo_unitario"
            ]

            * random.uniform(
                0.98,
                1.03
            ),

            2
        )


        lucro_bruto = round(
            valor_liquido
            - custo_total,
            2
        )


        margem_percentual = (

            round(
                lucro_bruto
                / valor_liquido,
                4
            )

            if valor_liquido > 0

            else 0
        )


        vendas_validas.append(
            {
                "id_item_venda":
                    f"ITV{len(vendas_validas) + 1:07d}",

                "id_venda":
                    f"VDA{numero_pedido:07d}",

                "data_venda":
                    data_venda,

                "id_cliente":
                    id_cliente,

                "id_produto":
                    id_produto,

                "id_vendedor":
                    id_vendedor,

                "quantidade":
                    quantidade,

                "preco_unitario":
                    preco_unitario,

                "valor_bruto":
                    valor_bruto,

                "percentual_desconto":
                    round(
                        percentual_desconto,
                        4
                    ),

                "valor_desconto":
                    valor_desconto,

                "valor_liquido":
                    valor_liquido,

                "custo_total":
                    custo_total,

                "lucro_bruto":
                    lucro_bruto,

                "margem_percentual":
                    margem_percentual
            }
        )


        total_pedido += (
            valor_liquido
        )


    pedidos_venda.append(
        {
            "id_venda":
                f"VDA{numero_pedido:07d}",

            "data_venda":
                data_venda,

            "id_cliente":
                id_cliente,

            "id_vendedor":
                id_vendedor,

            "valor_liquido":
                round(
                    total_pedido,
                    2
                )
        }
    )


print(
    f"Pedidos: "
    f"{len(pedidos_venda):,}"
)

print(
    f"Itens vendidos: "
    f"{len(vendas_validas):,}"
)

In [0]:
# ---------------------------------------------------------
# METAS
# ---------------------------------------------------------

def gerar_meses(
    inicio,
    fim
):

    atual = date(
        inicio.year,
        inicio.month,
        1
    )

    ultimo = date(
        fim.year,
        fim.month,
        1
    )


    meses = []


    while atual <= ultimo:

        meses.append(
            atual
        )

        if atual.month == 12:

            atual = date(
                atual.year + 1,
                1,
                1
            )

        else:

            atual = date(
                atual.year,
                atual.month + 1,
                1
            )


    return meses

In [0]:
metas_vendas = []


for mes in gerar_meses(
    DATA_INICIO,
    DATA_FIM
):

    numero_mes = (
        (mes.year - 2024) * 12
        + mes.month
        - 1
    )


    crescimento = (
        1
        + 0.012
        * numero_mes
    )


    sazonalidade = (
        fator_mes[
            mes.month
        ]
    )


    for vendedor in vendedores:

        id_vendedor = (
            vendedor[
                "id_vendedor"
            ]
        )


        valor_meta = (

            95_000

            * fator_desempenho_vendedor[
                id_vendedor
            ]

            * crescimento

            * sazonalidade

            * random.uniform(
                0.94,
                1.06
            )
        )


        metas_vendas.append(
            {
                "ano_mes":
                    mes.strftime(
                        "%Y-%m"
                    ),

                "id_vendedor":
                    id_vendedor,

                "valor_meta":
                    round(
                        valor_meta,
                        2
                    ),

                "meta_clientes":
                    int(
                        round(
                            50
                            * fator_desempenho_vendedor[
                                id_vendedor
                            ]
                            * random.uniform(
                                0.90,
                                1.10
                            )
                        )
                    )
            }
        )


print(
    f"Metas geradas: "
    f"{len(metas_vendas):,}"
)

In [0]:
# ---------------------------------------------------------
# COMPRAS
# ---------------------------------------------------------

mapa_fornecedores = {
    fornecedor["id_fornecedor"]:
        fornecedor

    for fornecedor
    in fornecedores
}


produtos_por_fornecedor = defaultdict(list)


for produto in produtos:

    if (
        produto[
            "situacao_produto"
        ]
        != "Ativo"
    ):

        continue


    produtos_por_fornecedor[
        produto[
            "id_fornecedor_principal"
        ]
    ].append(
        produto[
            "id_produto"
        ]
    )


fornecedores_com_produtos = list(
    produtos_por_fornecedor.keys()
)


peso_fornecedor = [

    sum(
        demanda_base[
            id_produto
        ]

        for id_produto

        in produtos_por_fornecedor[
            id_fornecedor
        ]
    )

    for id_fornecedor

    in fornecedores_com_produtos
]

In [0]:
compras_validas = []

pedidos_compra = []


QUANTIDADE_COMPRAS = 8_000


for numero_compra in range(
    1,
    QUANTIDADE_COMPRAS + 1
):

    data_compra = random.choices(
        datas,
        weights=pesos_datas,
        k=1
    )[0]


    id_fornecedor = random.choices(

        fornecedores_com_produtos,

        weights=peso_fornecedor,

        k=1

    )[0]


    produtos_disponiveis = (
        produtos_por_fornecedor[
            id_fornecedor
        ]
    )


    quantidade_itens = min(

        len(
            produtos_disponiveis
        ),

        random.choices(
            [1, 2, 3],
            weights=[28, 52, 20],
            k=1
        )[0]
    )


    produtos_escolhidos = (
        random.sample(
            produtos_disponiveis,
            quantidade_itens
        )
    )


    total_compra = 0


    for id_produto in (
        produtos_escolhidos
    ):

        produto = mapa_produtos[
            id_produto
        ]


        quantidade = max(

            20,

            int(
                demanda_base[
                    id_produto
                ]

                * random.uniform(
                    12,
                    35
                )
            )
        )


        custo_unitario = round(

            produto[
                "custo_unitario"
            ]

            * random.uniform(
                0.94,
                1.03
            ),

            2
        )


        valor_total = round(
            quantidade
            * custo_unitario,
            2
        )


        prazo_entrega = max(

            1,

            mapa_fornecedores[
                id_fornecedor
            ][
                "prazo_medio_entrega"
            ]

            + random.randint(
                -2,
                4
            )
        )


        recebimento_previsto = (
            data_compra

            + timedelta(
                days=prazo_entrega
            )
        )


        data_recebimento = (

            recebimento_previsto

            if (
                recebimento_previsto
                <= DATA_FIM
            )

            else None
        )


        compras_validas.append(
            {
                "id_item_compra":
                    f"ITC{len(compras_validas) + 1:07d}",

                "id_compra":
                    f"CMP{numero_compra:06d}",

                "data_compra":
                    data_compra,

                "id_fornecedor":
                    id_fornecedor,

                "id_produto":
                    id_produto,

                "quantidade":
                    quantidade,

                "custo_unitario":
                    custo_unitario,

                "valor_total":
                    valor_total,

                "prazo_entrega":
                    prazo_entrega,

                "data_recebimento":
                    data_recebimento
            }
        )


        total_compra += (
            valor_total
        )


    pedidos_compra.append(
        {
            "id_compra":
                f"CMP{numero_compra:06d}",

            "data_compra":
                data_compra,

            "id_fornecedor":
                id_fornecedor,

            "valor_total":
                round(
                    total_compra,
                    2
                )
        }
    )


print(
    f"Pedidos de compra: "
    f"{len(pedidos_compra):,}"
)

print(
    f"Itens de compra: "
    f"{len(compras_validas):,}"
)

In [0]:
# ---------------------------------------------------------
# CONTAS A RECEBER
# ---------------------------------------------------------

contas_receber_validas = []


for numero, pedido in enumerate(
    pedidos_venda,
    start=1
):

    id_cliente = pedido[
        "id_cliente"
    ]


    risco = (
        perfil_interno_cliente[
            id_cliente
        ][
            "risco_credito"
        ]
    )


    prazo = random.choices(
        [7, 15, 30, 45],
        weights=[
            10,
            20,
            50,
            20
        ],
        k=1
    )[0]


    data_vencimento = (

        pedido["data_venda"]

        + timedelta(
            days=prazo
        )
    )


    if data_vencimento > DATA_FIM:

        status = "A vencer"

        data_pagamento = None

        dias_atraso = 0


    else:

        probabilidade_vencido = {

            "Baixo": 0.015,
            "Médio": 0.05,
            "Alto": 0.14

        }[risco]


        probabilidade_atraso = {

            "Baixo": 0.08,
            "Médio": 0.18,
            "Alto": 0.35

        }[risco]


        sorteio = random.random()


        if (
            sorteio
            < probabilidade_vencido
        ):

            status = "Vencido"

            data_pagamento = None

            dias_atraso = (
                DATA_FIM
                - data_vencimento
            ).days


        elif (
            sorteio
            <
            probabilidade_vencido
            + probabilidade_atraso
        ):

            atraso_pagamento = (
                random.randint(
                    1,
                    60
                )
            )


            pagamento = (

                data_vencimento

                + timedelta(
                    days=atraso_pagamento
                )
            )


            if pagamento <= DATA_FIM:

                status = (
                    "Pago em atraso"
                )

                data_pagamento = (
                    pagamento
                )

                dias_atraso = (
                    atraso_pagamento
                )

            else:

                status = "Vencido"

                data_pagamento = None

                dias_atraso = (
                    DATA_FIM
                    - data_vencimento
                ).days


        else:

            antecipacao = (
                random.randint(
                    0,
                    5
                )
            )


            data_pagamento = (

                data_vencimento

                - timedelta(
                    days=antecipacao
                )
            )


            status = "Pago em dia"

            dias_atraso = 0


    contas_receber_validas.append(
        {
            "id_titulo_receber":
                f"REC{numero:07d}",

            "id_venda":
                pedido[
                    "id_venda"
                ],

            "id_cliente":
                id_cliente,

            "data_emissao":
                pedido[
                    "data_venda"
                ],

            "data_vencimento":
                data_vencimento,

            "data_pagamento":
                data_pagamento,

            "valor_titulo":
                pedido[
                    "valor_liquido"
                ],

            "status_titulo":
                status,

            "dias_atraso":
                dias_atraso
        }
    )


print(
    "Títulos a receber:",
    f"{len(contas_receber_validas):,}"
)

In [0]:
# ---------------------------------------------------------
# CONTAS A PAGAR
# ---------------------------------------------------------

contas_pagar_validas = []


for numero, compra in enumerate(
    pedidos_compra,
    start=1
):

    fornecedor = (
        mapa_fornecedores[
            compra[
                "id_fornecedor"
            ]
        ]
    )


    prazo = int(
        fornecedor[
            "condicao_pagamento"
        ].split()[0]
    )


    data_vencimento = (

        compra[
            "data_compra"
        ]

        + timedelta(
            days=prazo
        )
    )


    if data_vencimento > DATA_FIM:

        status = "A vencer"

        data_pagamento = None


    else:

        sorteio = random.random()


        if sorteio < 0.015:

            status = "Vencido"

            data_pagamento = None


        elif sorteio < 0.09:

            atraso = random.randint(
                1,
                20
            )


            pagamento = (

                data_vencimento

                + timedelta(
                    days=atraso
                )
            )


            if pagamento <= DATA_FIM:

                status = (
                    "Pago em atraso"
                )

                data_pagamento = (
                    pagamento
                )

            else:

                status = "Vencido"

                data_pagamento = None


        else:

            status = "Pago em dia"

            data_pagamento = (

                data_vencimento

                - timedelta(
                    days=random.randint(
                        0,
                        3
                    )
                )
            )


    contas_pagar_validas.append(
        {
            "id_titulo_pagar":
                f"PAG{numero:06d}",

            "id_compra":
                compra[
                    "id_compra"
                ],

            "id_fornecedor":
                compra[
                    "id_fornecedor"
                ],

            "data_emissao":
                compra[
                    "data_compra"
                ],

            "data_vencimento":
                data_vencimento,

            "data_pagamento":
                data_pagamento,

            "valor_titulo":
                compra[
                    "valor_total"
                ],

            "status_titulo":
                status
        }
    )


print(
    "Títulos a pagar:",
    f"{len(contas_pagar_validas):,}"
)

In [0]:
# ---------------------------------------------------------
# ESTOQUE
# ---------------------------------------------------------

dados_demanda_produto = [

    (
        produto["id_produto"],
        float(
            produto[
                "custo_unitario"
            ]
        ),
        float(
            demanda_base.get(
                produto[
                    "id_produto"
                ],
                1
            )
        )
    )

    for produto in produtos

    if produto[
        "situacao_produto"
    ] == "Ativo"
]


df_demanda_produto = (
    spark.createDataFrame(
        dados_demanda_produto,

        [
            "id_produto",
            "custo_medio",
            "demanda_media_dia"
        ]
    )
)

In [0]:
quantidade_dias = (
    DATA_FIM
    - DATA_INICIO
).days + 1


df_datas = (

    spark.range(
        quantidade_dias
    )

    .select(

        F.date_add(
            F.lit(
                DATA_INICIO.isoformat()
            ).cast("date"),

            F.col("id").cast("int")
        ).alias(
            "data_referencia"
        )
    )
)

In [0]:
df_estoque = (

    df_datas

    .crossJoin(
        df_demanda_produto
    )

    .withColumn(

        "estoque_minimo",

        F.round(
            F.col(
                "demanda_media_dia"
            )
            * 7
        ).cast("int")
    )

    .withColumn(

        "estoque_maximo",

        F.round(
            F.col(
                "demanda_media_dia"
            )
            * 30
        ).cast("int")
    )

    .withColumn(
        "fator_estoque",
        F.rand(
            SEMENTE + 10
        )
    )

    .withColumn(

        "quantidade_estoque",

        F.round(

            F.col(
                "estoque_minimo"
            )

            +

            (
                F.col(
                    "estoque_maximo"
                )

                - F.col(
                    "estoque_minimo"
                )
            )

            * F.col(
                "fator_estoque"
            )

        ).cast("int")
    )
)

In [0]:
df_estoque = (

    df_estoque

    .withColumn(

        "sorteio_ruptura",

        F.rand(
            SEMENTE + 20
        )
    )

    .withColumn(

        "quantidade_estoque",

        F.when(

            F.col(
                "sorteio_ruptura"
            )
            < 0.015,

            F.lit(0)

        ).otherwise(

            F.col(
                "quantidade_estoque"
            )
        )
    )
)

In [0]:
df_estoque = (

    df_estoque

    .withColumn(

        "sorteio_excesso",

        F.rand(
            SEMENTE + 30
        )
    )

    .withColumn(

        "quantidade_estoque",

        F.when(

            (
                F.col(
                    "demanda_media_dia"
                )
                < 8
            )

            &

            (
                F.col(
                    "sorteio_excesso"
                )
                < 0.08
            ),

            F.round(

                F.col(
                    "estoque_maximo"
                )

                * F.lit(1.6)

            ).cast("int")

        ).otherwise(

            F.col(
                "quantidade_estoque"
            )
        )
    )
)

In [0]:
df_estoque = (

    df_estoque

    .withColumn(

        "valor_estoque",

        F.round(

            F.col(
                "quantidade_estoque"
            )

            * F.col(
                "custo_medio"
            ),

            2
        )
    )

    .select(

        "data_referencia",
        "id_produto",
        "quantidade_estoque",
        "custo_medio",
        "valor_estoque",
        "estoque_minimo",
        "estoque_maximo"
    )
)


print(
    "Registros de estoque:",
    f"{df_estoque.count():,}"
)

In [0]:
# ---------------------------------------------------------
# PROBLEMAS CONTROLADOS DE QUALIDADE
# ---------------------------------------------------------

clientes_raw = [
    registro.copy()
    for registro in clientes
]


vendas_raw = [
    registro.copy()
    for registro in vendas_validas
]


compras_raw = [
    registro.copy()
    for registro in compras_validas
]


receber_raw = [
    registro.copy()
    for registro in contas_receber_validas
]


pagar_raw = [
    registro.copy()
    for registro in contas_pagar_validas
]


metas_raw = [
    registro.copy()
    for registro in metas_vendas
]

In [0]:
# Alguns estados em formato inconsistente

for indice in random.sample(
    range(len(clientes_raw)),
    8
):

    clientes_raw[indice][
        "estado"
    ] = (
        clientes_raw[indice][
            "estado"
        ].lower()
    )


# Espaços adicionais no nome da cidade

for indice in random.sample(
    range(len(clientes_raw)),
    12
):

    clientes_raw[indice][
        "cidade"
    ] = (
        clientes_raw[indice][
            "cidade"
        ]
        + " "
    )


# Alguns segmentos ausentes

for indice in random.sample(
    range(len(clientes_raw)),
    5
):

    clientes_raw[indice][
        "segmento_cliente"
    ] = None


# Registros duplicados

for indice in random.sample(
    range(len(clientes_raw)),
    8
):

    clientes_raw.append(
        clientes_raw[
            indice
        ].copy()
    )

In [0]:
# Algumas vendas sem vendedor

for indice in random.sample(
    range(len(vendas_raw)),
    25
):

    vendas_raw[indice][
        "id_vendedor"
    ] = None


# Algumas quantidades inconsistentes

for indice in random.sample(
    range(len(vendas_raw)),
    12
):

    vendas_raw[indice][
        "quantidade"
    ] = 0


# Duplicações

for indice in random.sample(
    range(len(vendas_raw)),
    30
):

    vendas_raw.append(
        vendas_raw[
            indice
        ].copy()
    )

In [0]:
# Pequenas inconsistências textuais

for indice in random.sample(
    range(len(receber_raw)),
    30
):

    receber_raw[indice][
        "status_titulo"
    ] = (
        receber_raw[indice][
            "status_titulo"
        ]
        + " "
    )


# Duplicações

for indice in random.sample(
    range(len(receber_raw)),
    10
):

    receber_raw.append(
        receber_raw[
            indice
        ].copy()
    )

In [0]:
for indice in random.sample(
    range(len(compras_raw)),
    15
):

    compras_raw.append(
        compras_raw[
            indice
        ].copy()
    )

In [0]:
# ---------------------------------------------------------
# DATAFRAMES
# ---------------------------------------------------------

df_fornecedores = spark.createDataFrame(
    fornecedores
)

df_produtos = spark.createDataFrame(
    produtos
)

df_vendedores = spark.createDataFrame(
    vendedores
)

df_clientes = spark.createDataFrame(
    clientes_raw
)

df_vendas = spark.createDataFrame(
    vendas_raw
)

df_compras = spark.createDataFrame(
    compras_raw
)

df_contas_receber = (
    spark.createDataFrame(
        receber_raw
    )
)

df_contas_pagar = (
    spark.createDataFrame(
        pagar_raw
    )
)

df_metas_vendas = (
    spark.createDataFrame(
        metas_raw
    )
)

In [0]:
# ---------------------------------------------------------
# FUNÇÃO DE GRAVAÇÃO
# ---------------------------------------------------------

def salvar_csv(
    dataframe,
    nome_fonte,
    numero_particoes=1
):

    caminho = (
        f"{caminho_volume}/"
        f"{nome_fonte}"
    )


    dataframe_saida = (
        dataframe.coalesce(
            numero_particoes
        )
    )


    (
        dataframe_saida

        .write

        .mode("overwrite")

        .option(
            "header",
            True
        )

        .option(
            "emptyValue",
            ""
        )

        .csv(
            caminho
        )
    )


    print(
        f"{nome_fonte:<20} "
        f"{dataframe.count():>10,} registros"
    )

In [0]:
salvar_csv(
    df_clientes,
    "clientes"
)

salvar_csv(
    df_produtos,
    "produtos"
)

salvar_csv(
    df_vendedores,
    "vendedores"
)

salvar_csv(
    df_fornecedores,
    "fornecedores"
)

salvar_csv(
    df_metas_vendas,
    "metas_vendas"
)


salvar_csv(
    df_vendas,
    "vendas",
    4
)


salvar_csv(
    df_compras,
    "compras",
    2
)


salvar_csv(
    df_contas_receber,
    "contas_receber",
    2
)


salvar_csv(
    df_contas_pagar,
    "contas_pagar",
    1
)


salvar_csv(
    df_estoque,
    "estoque",
    6
)

In [0]:
# ---------------------------------------------------------
# VALIDAÇÃO DOS ARQUIVOS GERADOS
# ---------------------------------------------------------

display(
    dbutils.fs.ls(
        caminho_volume
    )
)

In [0]:
resumo = [

    ("clientes", df_clientes.count()),

    ("produtos", df_produtos.count()),

    ("vendedores", df_vendedores.count()),

    ("fornecedores", df_fornecedores.count()),

    ("vendas", df_vendas.count()),

    ("compras", df_compras.count()),

    (
        "contas_receber",
        df_contas_receber.count()
    ),

    (
        "contas_pagar",
        df_contas_pagar.count()
    ),

    (
        "metas_vendas",
        df_metas_vendas.count()
    ),

    (
        "estoque",
        df_estoque.count()
    )
]


df_resumo = spark.createDataFrame(
    resumo,
    [
        "fonte",
        "quantidade_registros"
    ]
)


display(
    df_resumo.orderBy(
        F.desc(
            "quantidade_registros"
        )
    )
)

In [0]:
display(
    df_vendas.limit(20)
)

In [0]:
display(
    df_contas_receber
    .filter(
        F.col(
            "status_titulo"
        )
        .contains(
            "Vencido"
        )
    )
    .limit(20)
)